### Dataset Corrections

In [152]:
import numpy as np
import pandas as pd

df = pd.read_csv('../data/02_interim/02_dataset_without_leakage.csv', low_memory=False)

#### 1. Grid Position

The dataset saves the exit from pit-lane as zero, what will cause the ML model to learn: "Exit from pit-lane is better than exit from grid position 1". Because of this, the grid position 0 will be changed to max grid pos + 1, based on each race individually, since grid size varies from 1950 to nowadays.

In [153]:
df['grid'] = df['grid'].replace(0, np.nan)

max_grid_per_race = df.groupby('raceId')['grid'].transform('max') + 1

df['grid'] = df['grid'].fillna(max_grid_per_race)
df['grid'] = df['grid'].astype(int)

#### 2. Drop of irrelevant columns for learning
Now, columns without relevant info to the algorithm learn will be dropped. `constructor_url`, for example.

In [154]:
# Constructor Info
df = df.drop(['constructorRef', 'name_constructor', 'nationality_constructor', 'url_constructor'], axis=1)

# Driver Info
df = df.drop(['driverRef', 'number_driver', 'code', 'forename', 'surname', 'dob', 'nationality', 'url_driver'], axis=1)

# Race Info
df = df.drop(['name', 'time_race', 'url'], axis=1)

# FP info
df = df.drop(['fp1_time', 'fp1_date', 'fp2_time', 'fp2_date', 'fp3_time', 'fp3_date'], axis=1)

# Quali and Sprint info
df = df.drop(['quali_time', 'sprint_time'], axis=1)

# Results info
df = df.drop(['number'], axis=1)

df

,statusId,status,status_group,constructorId,driverId,raceId,year,round,circuitId,date,quali_date,sprint_date,resultId,grid,positionOrder,laps
0,1,Finished,Finished,1,1,2,2009,2,2,2009-04-05,\N,\N,7580,12,7,31
1,1,Finished,Finished,1,1,3,2009,3,17,2009-04-19,\N,\N,7599,9,6,56
2,1,Finished,Finished,1,1,4,2009,4,3,2009-04-26,\N,\N,7617,5,4,57
3,1,Finished,Finished,1,1,7,2009,7,5,2009-06-07,\N,\N,7686,16,13,58
4,1,Finished,Finished,1,1,10,2009,10,11,2009-07-26,\N,\N,7734,4,1,70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26754,140,Undertray,Others,3,849,1084,2022,11,70,2022-07-10,2022-07-08,2022-07-09,25624,17,19,48
26755,140,Undertray,Others,6,844,1111,2023,13,39,2023-08-27,2023-08-26,\N,26104,9,19,41
26756,140,Undertray,Others,117,4,1116,2023,18,69,2023-10-22,2023-10-20,2023-10-21,26201,17,16,49
26757,140,Undertray,Others,213,852,1085,2022,12,34,2022-07-24,2022-07-23,\N,25645,8,20,17


#### 3. Data Typing

In [155]:
df = df.replace(to_replace='\\N', value=np.nan)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26759 entries, 0 to 26758
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   statusId       26759 non-null  int64
 1   status         26759 non-null  str  
 2   status_group   26759 non-null  str  
 3   constructorId  26759 non-null  int64
 4   driverId       26759 non-null  int64
 5   raceId         26759 non-null  int64
 6   year           26759 non-null  int64
 7   round          26759 non-null  int64
 8   circuitId      26759 non-null  int64
 9   date           26759 non-null  str  
 10  quali_date     1799 non-null   str  
 11  sprint_date    360 non-null    str  
 12  resultId       26759 non-null  int64
 13  grid           26759 non-null  int64
 14  positionOrder  26759 non-null  int64
 15  laps           26759 non-null  int64
dtypes: int64(11), str(5)
memory usage: 3.3 MB


Fixing Column Types

In [156]:
df['date'] = pd.to_datetime(df['date'])

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26759 entries, 0 to 26758
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   statusId       26759 non-null  int64         
 1   status         26759 non-null  str           
 2   status_group   26759 non-null  str           
 3   constructorId  26759 non-null  int64         
 4   driverId       26759 non-null  int64         
 5   raceId         26759 non-null  int64         
 6   year           26759 non-null  int64         
 7   round          26759 non-null  int64         
 8   circuitId      26759 non-null  int64         
 9   date           26759 non-null  datetime64[us]
 10  quali_date     1799 non-null   str           
 11  sprint_date    360 non-null    str           
 12  resultId       26759 non-null  int64         
 13  grid           26759 non-null  int64         
 14  positionOrder  26759 non-null  int64         
 15  laps           26759 non-null 

#### 4. Temporal Ordering

In [157]:
df = df.sort_values(by=['date', 'raceId'])

df = df.reset_index(drop=True)

#### Saving updated dataset

In [159]:
df.to_csv('../data/02_interim/03_clean_races_info.csv', index=False)